# Partial pixel handling

In [ ]:
import numpy as np
from photutils.aperture import CircularAnnulus

from matplotlib import pyplot as plt

plt.style.use('../photutils_notebook_style.mplstyle')

There will always be pixels near the edge of an aperture or annulus that partially overlap with the aperture or annulus.

There are three ways that [`photutils`](https://photutils.readthedocs.io/en/stable/) can handle a pixel that is partialy with an aperture (everything that follows also applies to annulus:

1. Include the entire pixel if the *center* of the pixel fals within the aperture. [`photutils`](https://photutils.readthedocs.io/en/stable/) uses the string `center` for this method.
2. Weight the pixel by the *exact* fraction of the pixel's area that falls within the aperture. [`photutils`](https://photutils.readthedocs.io/en/stable/) uses the string `exact` for this method.
3. Break the pixel into *subpixels* where the value in each subpixel is interpolated from the image, and include the subpixels whose center is in the aperture. [`photutils`](https://photutils.readthedocs.io/en/stable/) uses the string `subpixel` for this meethod.

## Illustration of the difference between partial pixel methods

One of the nice features of `photutils` apertures is that they cn be converted to a mask and then displayed. While one typically would not do this as part of photometry it does clarify what each option means.

We begin by creating a single annulus and then convert it to a mask using each of the available options: `center`, `exact`, and `subpixel`. An annulus is used here because the partial pixel handling is visible in more places.

For the `subpixel` case one can specify how many subpixels should be used per side. The default is 5; the value used below was chosen to make clearer that there is a difference between `exact` and `subpixel`.

In [ ]:
# The position is not important in this case
aperture_position = [100, 100]

aperture_radius = 15

aperture = CircularAnnulus(aperture_position, 0.8 * aperture_radius, aperture_radius)

center_mask = aperture.to_mask(method="center")
exact_max = aperture.to_mask(method="exact")
subpix_mask = aperture.to_mask(method="subpixel", subpixels=2)

Let's see what these masks look like.

In [ ]:
fig, axs = plt.subplots(ncols=3, sharey=True)

mask_padding = 10

for ax, mask in zip(axs, [center_mask, exact_max, subpix_mask]):
    # Pad each aperture a bit so we can see the edges better
    mask_display = np.zeros(np.array(mask.shape) + mask_padding)
    start = mask_padding // 2
    end = start + mask.shape[0]
    mask_display[start:end, start:end] = mask
    ax.imshow(mask_display)
    ax.grid()
    aperture.plot(ax=ax, color="red")